In [0]:
import pyspark.sql.functions as F

In [0]:
RAW_FILE_PATH = "/Volumes/workspace/bronze/source_systems/kaggle/access.log"

In [0]:
BRONZE_TABLE_NAME = "workspace.bronze.raw_access_logs"
df_raw = spark.read.text(RAW_FILE_PATH)

#write data to the Bronze Layer

In [0]:
df_bronze = df_raw.select(
    F.col("value"),
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
BRONZE_TABLE_NAME = "bronze.raw_access_logs"

df_bronze.write.format("delta") \
    .mode("append") \
    .saveAsTable(BRONZE_TABLE_NAME)

print(f"Success! Raw data loaded into Bronze Delta Table: {BRONZE_TABLE_NAME}")


In [0]:
CSV_SOURCE_PATH = "/Volumes/workspace/bronze/source_systems/kaggle/client_hostname.csv"
CSV_BRONZE_TABLE = "bronze.client_lookup"

df_clients_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(CSV_SOURCE_PATH)

df_clients_bronze = df_clients_raw.select(
    "*", 
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingestion_timestamp")
)

df_clients_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(CSV_BRONZE_TABLE)

print(f"Success! Client lookup data loaded into: {CSV_BRONZE_TABLE}")
display(spark.read.table(CSV_BRONZE_TABLE).limit(5))